# Explorando modelos preentrenados: ResNet, EfficientNet y más

En este notebook mostraremos que el ecosistema de modelos preentrenados no se limita a MobileNetV2.

Trabajaremos principalmente con:

- **ResNet50**
- **EfficientNetB0**

y veremos ejemplos de:

- predicción con modelos preentrenados,
- activaciones intermedias,
- **Grad-CAM**,
- y, como bonus, una carga de modelo desde **Hugging Face**

## Objetivos de este notebook

Al finalizar este notebook deberías poder:

- reconocer ideas básicas detrás de **ResNet** y **EfficientNet**,
- cargar modelos preentrenados desde Keras,
- comparar arquitecturas a nivel práctico,
- visualizar salidas intermedias,
- aplicar Grad-CAM,
- y entender que también existen otras fuentes de modelos preentrenados, como Hugging Face.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print("TensorFlow version:", tf.__version__)

## ¿Qué es [ResNet](https://arxiv.org/pdf/1512.03385)?

**ResNet** es una familia de redes profundas muy influyente.

La idea central es usar **conexiones residuales** o *skip connections*.

En vez de obligar a un bloque a aprender toda una transformación desde cero, se le permite aprender una corrección sobre la entrada.

### Intuición
Eso ayuda a que redes más profundas se entrenen mejor y facilita el flujo de información y del gradiente.

## ¿Qué es [EfficientNet](https://arxiv.org/pdf/1905.11946)?

**EfficientNet** es una familia de modelos que busca un mejor equilibrio entre:

- precisión,
- tamaño del modelo,
- y costo computacional.

La idea general es escalar el modelo de forma más balanceada, en vez de hacerlo crecer de manera arbitraria solo en profundidad o solo en ancho.

En este notebook usaremos **EfficientNetB0**, que es una versión relativamente liviana y práctica para demostraciones.

## Modelos preentrenados en Keras

Keras Applications ofrece modelos preentrenados que pueden usarse para:

- predicción,
- extracción de características,
- fine-tuning.

La lógica general de uso será similar a la que ya vimos con MobileNetV2.

In [ ]:
from tensorflow.keras.applications import ResNet50, EfficientNetB0
from tensorflow.keras.applications.resnet import preprocess_input as resnet_preprocess
from tensorflow.keras.applications.resnet import decode_predictions as resnet_decode
from tensorflow.keras.applications.efficientnet import preprocess_input as efficientnet_preprocess
from tensorflow.keras.applications.efficientnet import decode_predictions as efficientnet_decode

print("Importaciones listas.")

## Cargar modelos preentrenados

Aquí los cargaremos con:

- `weights="imagenet"` para usar pesos preentrenados,
- `include_top=True` para mantener la cabeza original de [clasificación de ImageNet](https://deeplearning.cms.waikato.ac.nz/user-guide/class-maps/IMAGENET/).

Eso nos servirá para hacer inferencia directa sobre una imagen.

In [ ]:
resnet_model = ResNet50(weights="imagenet", include_top=True)
efficientnet_model = EfficientNetB0(weights="imagenet", include_top=True)

print("ResNet50 cargada.")
print("EfficientNetB0 cargada.")

## Comparación rápida

Veamos algunas diferencias simples entre ambos modelos.

In [ ]:
print("Nombre modelo 1:", resnet_model.name)
print("Parámetros ResNet50:", resnet_model.count_params())
print("Output shape ResNet50:", resnet_model.output_shape)

print()

print("Nombre modelo 2:", efficientnet_model.name)
print("Parámetros EfficientNetB0:", efficientnet_model.count_params())
print("Output shape EfficientNetB0:", efficientnet_model.output_shape)

## Cargar una imagen de ejemplo

Usaremos una imagen desde una URL pública para mostrar predicción y visualización.

In [ ]:
img_url = "https://storage.googleapis.com/download.tensorflow.org/example_images/592px-Red_sunflower.jpg"
img_path = keras.utils.get_file("example_image.jpg", origin=img_url)

img = keras.utils.load_img(img_path, target_size=(224, 224))
img_array = keras.utils.img_to_array(img)

plt.figure(figsize=(5, 5))
plt.imshow(img)
plt.axis("off")
plt.show()

## Preparar batch

Los modelos esperan batches, así que agregaremos una dimensión extra.

In [ ]:
img_batch = np.expand_dims(img_array, axis=0)
print("Shape del batch:", img_batch.shape)

## Predicción con ResNet50

Cada familia de modelos puede requerir su propio preprocesamiento.

In [ ]:
resnet_input = resnet_preprocess(img_batch.copy())
resnet_preds = resnet_model.predict(resnet_input, verbose=0)

print("Top-5 predicciones ResNet50:")
for pred in resnet_decode(resnet_preds, top=5)[0]:
    print(pred)

Usamos `img_batch.copy()` para conservar intacta la imagen original.
Así, el preprocesamiento para ResNet se aplica sobre una copia y no sobre el batch original.

## Predicción con EfficientNetB0

In [ ]:
efficientnet_input = efficientnet_preprocess(img_batch.copy())
efficientnet_preds = efficientnet_model.predict(efficientnet_input, verbose=0)

print("Top-5 predicciones EfficientNetB0:")
for pred in efficientnet_decode(efficientnet_preds, top=5)[0]:
    print(pred)

## Observación importante

Aunque ambos modelos están preentrenados en ImageNet, no debemos asumir que todo es idéntico entre familias.

Siempre conviene revisar:

- el tamaño de entrada,
- el preprocesamiento esperado,
- el costo computacional,
- y la arquitectura.

## Visualizar activaciones intermedias

Un modelo preentrenado no solo sirve para entregar una clase final.

También podemos inspeccionar qué tipo de representaciones aparecen dentro de la red.

Aquí construiremos un modelo auxiliar para extraer la salida de una capa intermedia de ResNet50.

In [ ]:
for i, layer in enumerate(resnet_model.layers[:20]):
    print(i, layer.name, layer.__class__.__name__)

## Elegir una capa intermedia

Tomaremos una capa relativamente temprana para visualizar algunos mapas de activación.

In [ ]:
layer_name = "conv1_relu"

activation_model = keras.Model(
    inputs=resnet_model.input,
    outputs=resnet_model.get_layer(layer_name).output
)

activations = activation_model.predict(resnet_input, verbose=0)
print("Shape activations:", activations.shape)

## Visualización de algunos feature maps

Mostraremos algunos canales de la salida intermedia.

No todos serán fácilmente interpretables, pero sirven para ver que la red responde a distintos patrones.

In [ ]:
n_filters = 9

plt.figure(figsize=(10, 10))
for i in range(n_filters):
    ax = plt.subplot(3, 3, i + 1)
    plt.imshow(activations[0, :, :, i], cmap="viridis")
    plt.title(f"Canal {i}")
    plt.axis("off")
plt.show()

## ¿Qué estamos viendo?

Estas imágenes no son “predicciones” finales.

Son **mapas de respuesta internos** que muestran dónde ciertos filtros de la red activan más intensamente.

En capas tempranas, muchas veces aparecen respuestas relacionadas con:

- bordes,
- contrastes,
- texturas,
- patrones simples.

## [Grad-CAM](https://arxiv.org/pdf/1610.02391)

Ahora mostraremos una técnica de interpretabilidad muy conocida: **Grad-CAM**.

La idea general es construir un mapa de calor que indique qué regiones de la imagen fueron más relevantes para la clase predicha.

Usaremos una versión simplificada de la lógica del ejemplo oficial de Keras.

## Grad-CAM: idea general

**Grad-CAM** significa **Gradient-weighted Class Activation Mapping**.

Es una técnica de interpretabilidad visual que busca responder una pregunta como esta:

> ¿Qué regiones de la imagen fueron más relevantes para que el modelo predijera esta clase?

La idea no es cambiar el modelo, sino analizar cómo una **clase específica** depende de las activaciones de una **capa convolucional profunda**.

Por eso Grad-CAM suele aplicarse sobre la **última capa convolucional** de una CNN:

- esa capa todavía conserva estructura espacial,
- pero ya contiene información semántica más rica que las capas tempranas.

## ¿Qué objetos intervienen en Grad-CAM?

En una versión simplificada, Grad-CAM combina dos cosas:

1. **Los feature maps** de una capa convolucional profunda  
   Estos mapas indican qué patrones están activados en distintas zonas de la imagen.

2. **Los gradientes de la clase objetivo respecto de esos feature maps**  
   Estos gradientes indican qué tan importante es cada mapa para esa clase.

### Intuición
- Los **feature maps** dicen: “qué patrones fueron detectados y dónde”.
- Los **gradientes** dicen: “qué tan útil fue cada uno de esos patrones para la clase que nos interesa”.

## Paso 1: elegir una clase de interés

Supongamos que el modelo predice una clase, por ejemplo:

- "golden retriever"
- o "tabby cat"

Grad-CAM trabaja con el **score** de esa clase específica.

Es decir, no analiza la salida completa del modelo al mismo tiempo, sino una componente puntual de la salida:

- la clase predicha,
- o cualquier otra clase que queramos inspeccionar.

### Idea importante
Grad-CAM responde a algo del tipo:

> “¿Qué partes de la imagen empujaron al modelo hacia esta clase?”

## Paso 2: calcular gradientes respecto de la última capa convolucional

Luego calculamos el gradiente de la salida de esa clase con respecto a la salida de la última capa convolucional.

En forma conceptual, queremos algo como:

$\frac{\partial y^c}{\partial A^k}$

donde:

- $y^c$ es el score de la clase $c$,
- $A^k$ es el feature map $k$ de la última capa convolucional.

### Interpretación
Este gradiente nos dice cuánto influye cada mapa de activación en la clase que estamos analizando.

## Paso 3: obtener un peso para cada feature map

Grad-CAM no usa directamente todos los gradientes píxel por píxel.

En cambio, para cada feature map $k$, calcula un único peso promedio:

$
\alpha_k^c = \frac{1}{Z} \sum_i \sum_j \frac{\partial y^c}{\partial A_{ij}^k}
$

donde:

- $A_{ij}^k$ es la activación en la posición $(i,j)$ del mapa $k$,
- $Z$ es el número total de posiciones espaciales.

### Intuición
Este promedio resume qué tan importante fue el mapa $k$ para la clase $c$.

Si el peso es grande y positivo, ese mapa fue útil para apoyar esa clase.

## Paso 4: combinación ponderada de los feature maps

Una vez obtenidos los pesos, Grad-CAM construye un mapa final combinando los feature maps:

$
L_{\text{Grad-CAM}}^c = \text{ReLU}\left( \sum_k \alpha_k^c A^k \right)
$

### ¿Qué significa esto?

- Cada mapa $A^k$ aporta según su peso $\alpha_k^c$.
- Luego se suman todos.
- Finalmente se aplica **ReLU**.

### ¿Por qué aparece ReLU?
Porque Grad-CAM busca resaltar principalmente las regiones que **apoyan** la clase de interés.

Las contribuciones negativas no son el foco principal de esta visualización.

## ¿Por qué usar la última capa convolucional?

Esta elección es importante.

Si usamos una capa muy temprana:

- tendremos mucha información espacial,
- pero todavía muy poca información semántica.

Si usamos una capa muy tardía, después de operaciones que eliminan estructura espacial:

- ya no podremos localizar bien dónde ocurrió la evidencia visual.

### Por eso se suele usar la última capa convolucional
Porque ofrece un buen equilibrio entre:

- localización espacial,
- y contenido semántico.

In [12]:
last_conv_layer_name = "conv5_block3_out"

In [13]:
def make_gradcam_heatmap(img_array, model, last_conv_layer_name, pred_index=None):
    grad_model = keras.models.Model(
        [model.inputs],
        [model.get_layer(last_conv_layer_name).output, model.output]
    )

    with tf.GradientTape() as tape:
        last_conv_layer_output, preds = grad_model(img_array)
        if pred_index is None:
            pred_index = tf.argmax(preds[0])
        class_channel = preds[:, pred_index]

    grads = tape.gradient(class_channel, last_conv_layer_output)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    last_conv_layer_output = last_conv_layer_output[0]
    heatmap = last_conv_layer_output @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)

    heatmap = tf.maximum(heatmap, 0) / tf.math.reduce_max(heatmap)
    return heatmap.numpy()

## Relación entre la teoría y el código

En el código de Grad-CAM aparecen justamente estos pasos:

1. crear un modelo auxiliar que entregue:
   - la salida de la última capa convolucional,
   - y la predicción final;

2. usar `GradientTape` para calcular gradientes;

3. promediar los gradientes sobre las dimensiones espaciales;

4. usar esos promedios como pesos;

5. combinar los feature maps con esos pesos;

6. aplicar una rectificación y normalizar el resultado para visualizarlo.

In [ ]:
heatmap = make_gradcam_heatmap(resnet_input, resnet_model, last_conv_layer_name)

plt.figure(figsize=(6, 6))
plt.imshow(heatmap, cmap="jet")
plt.colorbar()
plt.title("Grad-CAM heatmap")
plt.axis("off")
plt.show()

## ¿Qué representa el heatmap final?

El heatmap de Grad-CAM no es una segmentación exacta del objeto.

Tampoco significa que el modelo “solo miró” esa región.

Lo que representa es algo más específico:

> una estimación de qué zonas de la imagen contribuyeron más a la clase analizada.

Por eso Grad-CAM debe interpretarse como una herramienta de apoyo para entender el modelo, no como una explicación perfecta o completa.

## Superponer el heatmap sobre la imagen original

In [ ]:
heatmap_resized = tf.image.resize(heatmap[..., np.newaxis], (224, 224)).numpy().squeeze()

plt.figure(figsize=(6, 6))
plt.imshow(img_array.astype("uint8"))
plt.imshow(heatmap_resized, cmap="jet", alpha=0.4)
plt.title("Grad-CAM sobre imagen original")
plt.axis("off")
plt.show()

## Interpretación

Grad-CAM no “explica todo” lo que hace el modelo, pero sí entrega una pista visual de:

- dónde estaba mirando,
- qué regiones influyeron más,
- y si el foco parece razonable o no.

## Mensaje práctico sobre Grad-CAM

Grad-CAM es útil porque:

- no requiere reentrenar el modelo,
- se puede aplicar después de entrenar,
- y permite visualizar de forma intuitiva dónde está la evidencia más relevante.

Pero también tiene límites:

- depende de la capa elegida,
- no ofrece una explicación causal completa,
- y puede ser sensible a detalles del modelo y de la imagen.

### Idea final
Grad-CAM no “abre completamente la caja negra”, pero sí entrega una pista visual muy valiosa sobre el proceso de decisión del modelo.

## Bonus 1: cargar un modelo desde Hugging Face

También existen modelos preentrenados fuera de Keras Applications.

Una fuente muy importante es **[Hugging Face](https://huggingface.co)**, donde es común cargar modelos con `from_pretrained()`.

En visión, una opción natural es un **Vision Transformer (ViT)**. La documentación oficial de Hugging Face lo presenta como un modelo de clasificación de imágenes y muestra el uso de `from_pretrained()`.

In [16]:
!pip -q install transformers pillow

In [17]:
from transformers import AutoImageProcessor, ViTForImageClassification
from PIL import Image
import requests

In [ ]:
hf_model_name = "google/vit-base-patch16-224"

processor = AutoImageProcessor.from_pretrained(hf_model_name)
hf_model = ViTForImageClassification.from_pretrained(hf_model_name)

print("Modelo Hugging Face cargado:", hf_model_name)

In [ ]:
pil_img = Image.open(img_path).convert("RGB")
inputs = processor(images=pil_img, return_tensors="pt")

outputs = hf_model(**inputs)
logits = outputs.logits
predicted_class_idx = logits.argmax(-1).item()

print("Clase predicha:", hf_model.config.id2label[predicted_class_idx])

## Bonus 2: modelos desde GitHub

Además de Keras Applications y Hugging Face, también existen repositorios de GitHub
con implementaciones de modelos preentrenados.

Sin embargo, estas opciones pueden requerir dependencias adicionales y ser más sensibles
a cambios de versión del ecosistema.

### Mensaje práctico
Para un prototipo rápido, Keras Applications y Hugging Face suelen ser opciones
más estables. GitHub ofrece más flexibilidad, pero también más riesgo técnico.